# QC confound 04 — Does the result survive equal sequencing depth?

Depth changes the joint shape of the QC metrics: in shallow libraries, gene count
saturates differently against UMI count and the low-count tail thickens. Because PBS
subtypes are defined by co-occurring extremes, depth can move PBS composition even
with per-specimen quantiles.

Intervention: **binomial thinning**. Each UMI in a specimen is kept with probability
`target / depth`, where depth is the median UMI count of that specimen's retained
cells. This simulates sequencing every library to the same depth while preserving
between-cell differences (unlike per-cell downsampling). Specimens already below the
target are left unchanged.

Pal's thresholds were tuned to each library's original depth, so after thinning, noise
is redefined with the per-specimen MAD rule from notebook 03. The comparison that
isolates depth is therefore **MAD on original counts vs MAD on depth-matched counts**.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from signals_in_the_noise.analysis.noise_phenotypes import ISCB_SCS_PBS_THRESHOLDS
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.preprocessing.gse161529 import GSE161529
from signals_in_the_noise.preprocessing.specimens import Cohort
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

POSITIVE, NEGATIVE = "ER+ tumour", "Normal"
SEED = 0
OUTPUT_DIR = get_data_path("processed") / "qc_confound"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
gse = GSE161529()
cohort = Cohort.from_objects(gse.objects.values(), [POSITIVE, NEGATIVE])
cohort.overview().groupby("condition").agg(specimens=("n_cells", "size"), cells=("n_cells", "sum"))

In [ ]:
from signals_in_the_noise.modeling.datasets import build_cell_table, standard_arms
from signals_in_the_noise.modeling.experiments import PUBLISHED_PROTOCOL, reference_specimens

reference_arm = standard_arms(ISCB_SCS_PBS_THRESHOLDS, names=["noise_pbs"])[0]
reference_table = build_cell_table(
    cohort.obs, cohort.conditions, reference_arm, patients=cohort.patients
)
included = reference_specimens(reference_table, PUBLISHED_PROTOCOL["min_reference_cells"])
min_cells = PUBLISHED_PROTOCOL["min_reference_cells"]
print(f"{len(included)} specimens have >= {min_cells} PBS-labelled noise cells")

In [ ]:
from signals_in_the_noise.preprocessing.qc import (
    depth_matched_qc_obs,
    library_depth,
    mad_thresholds,
    relabel_noise,
)

N_MADS = 3.0
TARGET_QUANTILE = 0.0  # 0.0 = shallowest included specimen

retained_masks = {
    sid: adata.obs["is_noise"].to_numpy() == 0 for sid, adata in cohort.specimens.items()
}
depths = pd.Series(
    {sid: library_depth(adata, retained_masks[sid]) for sid, adata in cohort.specimens.items()}
)
target_depth = float(depths.loc[included].quantile(TARGET_QUANTILE))
print(f"target median UMI per retained cell: {target_depth:.0f}")

In [ ]:
rng = np.random.default_rng(SEED)
thinned_obs, fractions = {}, {}
for sid, adata in cohort.specimens.items():
    thinned_obs[sid], fractions[sid] = depth_matched_qc_obs(
        adata, target_depth, rng, reference_mask=retained_masks[sid]
    )

depth_table = pd.DataFrame(
    {
        "condition": pd.Series(cohort.conditions),
        "depth": depths,
        "thinning_fraction": pd.Series(fractions),
    }
)
depth_table["included"] = depth_table.index.isin(included)
depth_table.sort_values(["condition", "depth"]).round(3)

In [ ]:
RULES = {"mad": "is_noise_mad"}


def mad_rule(obs):
    return mad_thresholds(obs, n_mads=N_MADS)


versions = {
    "original": relabel_noise(cohort.obs, mad_rule, RULES["mad"]),
    "depth-matched": relabel_noise(thinned_obs, mad_rule, RULES["mad"]),
}

## 1. Sanity check: did thinning remove the depth difference?

After thinning, `median_log1p_total_counts_retained` should no longer separate the
conditions (AUC near 0.5) among included specimens.

In [ ]:
from signals_in_the_noise.analysis.confounders import association_table, covariates_from_obs
from signals_in_the_noise.utils.visualization import plot_specimen_qc_distributions

depth_columns = [
    "median_log1p_total_counts_retained",
    "median_log1p_n_genes_by_counts_retained",
    "median_complexity_retained",
    "noise_fraction",
]
checks = {}
for version, obs in versions.items():
    covs = covariates_from_obs({sid: obs[sid] for sid in included}, cohort.conditions, RULES["mad"])
    checks[version] = association_table(
        covs, positive=POSITIVE, negative=NEGATIVE, columns=depth_columns
    )
depth_check = pd.concat(checks, names=["version", "covariate"])
display(depth_check.round(4))

plot_specimen_qc_distributions(
    {sid: versions["depth-matched"][sid] for sid in included},
    cohort.conditions,
    ["log1p_total_counts", "log1p_n_genes_by_counts"],
    population="all",
    noise_column=RULES["mad"],
)
plt.show()

## 2. Core arms on original vs depth-matched counts

In [ ]:
from signals_in_the_noise.modeling.datasets import CORE_ARM_NAMES, standard_arms
from signals_in_the_noise.modeling.experiments import (
    PUBLISHED_PROTOCOL,
    InsufficientSpecimensError,
    run_arm_suite,
)

N_PERMUTATIONS = 200
results = {}
for version, obs in versions.items():
    try:
        results[version] = run_arm_suite(
            obs,
            cohort.conditions,
            standard_arms(ISCB_SCS_PBS_THRESHOLDS, names=CORE_ARM_NAMES),
            positive_label=POSITIVE,
            negative_label=NEGATIVE,
            patients=cohort.patients,
            noise_column=RULES["mad"],
            n_permutations=N_PERMUTATIONS,
            seed=SEED,
            **PUBLISHED_PROTOCOL,
        )
    except InsufficientSpecimensError as error:
        print(f"{version}: skipped ({error})")
        continue
    print(f"{version}: {len(results[version].specimens)} specimens included")
assert results, "Neither count version left enough specimens to compare conditions."

In [ ]:
summary = pd.concat(
    {version: r.summary() for version, r in results.items()}, names=["version", "arm"]
)
display(
    summary[
        [
            "n_specimens",
            "specimen_auc",
            "permutation_null_mean",
            "permutation_p",
            "draws_mean_cell_accuracy",
            "draws_mean_specimen_auc",
        ]
    ].round(3)
)
paired = pd.concat(
    {version: r.paired_vs_reference("cell_accuracy") for version, r in results.items()},
    names=["version", "arm"],
)
display(paired.round(3))

ax = sns.pointplot(
    data=summary.reset_index(),
    x="version",
    y="specimen_auc",
    hue="arm",
    dodge=0.3,
    linestyle="none",
)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=1)
ax.set_title("Specimen AUC original vs depth-matched")
plt.show()

In [ ]:
depth_table.to_csv(OUTPUT_DIR / "04_depth_and_thinning.csv")
depth_check.to_csv(OUTPUT_DIR / "04_depth_sanity_check.csv")
summary.to_csv(OUTPUT_DIR / "04_summary_by_version.csv")
paired.to_csv(OUTPUT_DIR / "04_paired_by_version.csv")

## Reading the results

| Observation | Meaning |
|---|---|
| Depth no longer separates conditions after thinning, and `noise_pbs` is unchanged | Sequencing depth does not explain the PBS signal. |
| `noise_pbs` drops toward its permutation null after thinning | Depth was driving PBS composition. |
| `retained_qc` collapses after thinning but `noise_pbs` does not | The raw QC difference was depth; the PBS signal is something else. |

Thinning is a single random realization; if the conclusion hinges on a borderline
p-value, re-run with a different `SEED`.